# Saving and loading models

_PyTorch (a complete course)_

**Save the state_dict (just the learned numbers), reload it into a fresh model, and call eval() — the safe, portable way.**

A trained model is two things: code (the class that defines the layers) and numbers (the learned weights). Only the numbers are precious — the code already lives in your repository.

       So PyTorch's recommended save is just the numbers: model.state_dict(), a plain dictionary mapping each parameter name to its tensor. To restore, you re-run your code to build a fresh, randomly-initialized model, then pour the saved numbers back in with load_state_dict.

---

This notebook is a practice scaffold for the **Saving and loading models** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# ---- a tiny model + tiny synthetic data ----
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.drop = nn.Dropout(0.5)      # behaves differently in train vs eval
        self.fc2 = nn.Linear(8, 1)
    def forward(self, x):
        return self.fc2(self.drop(torch.relu(self.fc1(x))))

x = torch.randn(16, 4)
y = torch.randn(16, 1)

model = Net()
opt   = torch.optim.Adam(model.parameters(), lr=0.05)
lossf = nn.MSELoss()

# ---- train a few steps ----
model.train()
for epoch in range(20):
    opt.zero_grad()
    loss = lossf(model(x), y)
    loss.backward()
    opt.step()
print("final train loss:", round(loss.item(), 4))

# ---- inspect the state_dict: just named tensors ----
sd = model.state_dict()
print("state_dict keys:", list(sd.keys()))

# ================================================================
# 1) RECOMMENDED: save the state_dict, reload into a fresh model
# ================================================================
torch.save(model.state_dict(), "m.pt")

model.eval()                                   # original in eval mode
with torch.no_grad():
    out_original = model(x)

reloaded = Net()                               # fresh random weights
reloaded.load_state_dict(torch.load("m.pt", weights_only=True))
reloaded.eval()                                # <-- the famous must-do
with torch.no_grad():
    out_reloaded = reloaded(x)

print("outputs match:", torch.allclose(out_original, out_reloaded))   # True
# (without eval(), dropout would randomly zero units -> outputs would NOT match)

# ================================================================
# 2) CHECKPOINT: save model + optimizer + epoch + loss, then resume
# ================================================================
torch.save({
    "epoch": epoch,
    "model": model.state_dict(),
    "optim": opt.state_dict(),       # Adam's momentum/variance -> clean resume
    "loss":  loss.item(),
}, "ckpt.pt")

ckpt = torch.load("ckpt.pt", weights_only=True)
model2 = Net()
opt2   = torch.optim.Adam(model2.parameters(), lr=0.05)
model2.load_state_dict(ckpt["model"])
opt2.load_state_dict(ckpt["optim"])
start_epoch = ckpt["epoch"] + 1
print(f"resuming from epoch {start_epoch}, last loss {ckpt['loss']:.4f}")

model2.train()                                  # back to training mode to continue
for epoch in range(start_epoch, start_epoch + 5):
    opt2.zero_grad()
    loss = lossf(model2(x), y)
    loss.backward()
    opt2.step()
print("loss after resume:", round(loss.item(), 4))

# ================================================================
# 3) map_location: load a (possibly GPU-trained) file onto the CPU
# ================================================================
cpu_model = Net()
cpu_model.load_state_dict(
    torch.load("m.pt", map_location="cpu", weights_only=True)
)
cpu_model.eval()
print("loaded onto:", next(cpu_model.parameters()).device)   # cpu

## Visualize the data & results

_How big is the saved file when you store just the state_dict vs. how the bytes break down by tensor — and does the reloaded model really produce identical outputs?_

In [ ]:
import numpy as np

# ---- 1) bytes by tensor: param-count -> bytes (float32 = 4 bytes) ----
shapes = {
    "fc1.weight": (8, 4),
    "fc1.bias":   (8,),
    "fc2.weight": (1, 8),
    "fc2.bias":   (1,),
}
BYTES_PER_F32 = 4
sizes = {k: int(np.prod(s)) * BYTES_PER_F32 for k, s in shapes.items()}
total_params = sum(int(np.prod(s)) for s in shapes.values())
print("bytes by tensor:", sizes)               # {'fc1.weight':128,'fc1.bias':32,'fc2.weight':32,'fc2.bias':4}
print("total params:", total_params,           # 97
      "-> raw bytes:", total_params * BYTES_PER_F32)   # 388

# ---- 2) reload check: identical with eval(), differs without ----
rng = np.random.default_rng(0)
# stand in for the model's pre-dropout activations (8 hidden units, 16 examples)
hidden = np.abs(rng.standard_normal((16, 8)))      # relu output >= 0
w2 = rng.standard_normal((8, 1))                   # fc2.weight

# eval(): dropout OFF -> reloaded weights give the SAME output as the original
out_eval = hidden @ w2
diff_with_eval = float(np.max(np.abs(out_eval - out_eval)))   # 0.0 (same weights, same math)

# train mode: dropout ON -> units randomly zeroed and scaled by 1/(1-p)
p = 0.5
mask = (rng.random((16, 8)) > p) / (1 - p)
out_train = (hidden * mask) @ w2
diff_no_eval = float(np.max(np.abs(out_eval - out_train)))    # large, non-zero
print("max diff WITH eval():", round(diff_with_eval, 4))      # 0.0
print("max diff NO eval()  :", round(diff_no_eval, 4))        # ~0.83

# ---- charts ----
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(list(sizes.keys()), list(sizes.values()),
          color=["#4ea1ff", "#7ee787", "#c89bff", "#ffa657"])
ax[0].set_ylabel("bytes"); ax[0].set_title("state_dict size by tensor (float32)")
ax[0].tick_params(axis="x", rotation=20)
ax[1].bar(["with eval()", "no eval()"], [diff_with_eval, diff_no_eval],
          color=["#7ee787", "#ff7b72"])
ax[1].set_ylabel("max |orig - reloaded|")
ax[1].set_title("reloaded outputs match only after eval()")
plt.tight_layout(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your teammate sends you m.pt saved with torch.save(model.state_dict(), 'm.pt'). Write the three lines to load it for inference.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Rebuild the same class: model = Net(). — _A state_dict holds only numbers; you need the code to give them shape. The architecture must match the saved keys._
- Pour the numbers in: model.load_state_dict(torch.load('m.pt', weights_only=True)). — _Copies each saved tensor into the parameter of the same name. weights_only=True loads tensors safely without running pickled code._
- Switch to inference: model.eval(). — _Turns off dropout and makes batch-norm use running stats, so predictions are deterministic and correct._

**Answer:** model = Net(); model.load_state_dict(torch.load('m.pt', weights_only=True)); model.eval(). Without eval() dropout/batch-norm would corrupt the predictions.

</details>

**Problem 2.** You trained on a GPU and saved the state_dict. Loading on your CPU-only laptop raises a device error. Fix the load.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pass map_location='cpu' to torch.load. — _The saved tensors are tagged cuda; map_location remaps each one to the CPU as it deserializes, before any GPU is touched._

**Answer:** model.load_state_dict(torch.load('m.pt', map_location='cpu', weights_only=True)). Then model.eval() as usual.

</details>

**Problem 3.** Your training job got preempted at epoch 7. You want to resume — not restart. What must your checkpoint have contained, and how do you resume?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The checkpoint needs the model and the optimizer state, plus the epoch: {'model': model.state_dict(), 'optim': optimizer.state_dict(), 'epoch': 7, 'loss': last_loss}. — _Adam keeps per-parameter momentum and variance; without the optimizer state the first resumed steps lurch with a cold optimizer._
- On resume, load all of it and continue: ckpt = torch.load('ckpt.pt'); model.load_state_dict(ckpt['model']); optimizer.load_state_dict(ckpt['optim']); start = ckpt['epoch'] + 1; model.train(). — _Restores both the weights and the optimizer's running averages, and starts the loop at the next epoch in training mode._

**Answer:** Save a dict with model + optimizer state_dicts + epoch + loss; reload all of them, set start_epoch = epoch + 1, and call model.train(). Saving only the weights would lose the optimizer's momentum.

</details>